# 🧹 Phase 1: Data Cleaning & Pre-processing

Based on the initial analysis, this notebook implements a **conservative cleaning strategy**.
We aim to fix structural errors (missing IDs) and optimize data types while preserving as much information as possible (e.g., keeping anime with partial data).

**Key Actions:**
1.  **Impute** missing values where logic allows (e.g., `favorites=0`, `is_rewatching=False`).
2.  **Drop** only structurally corrupted rows (missing Primary/Foreign Keys).
3.  **Extract** useful features (e.g., `year` from `start_date`).
4.  **Optimize** data types (float64 -> float32/16, objects -> categories) to reduce memory usage.
5.  **Save** to optimized Parquet format for the next steps.

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import gc
from pathlib import Path
import pprint

# 🧪 Standard Import (requires 'solution' package installed)
from solution.src.common import PROCESSED_DATA_DIR
from solution.src.data import load_raw, get_raw_path, PROCESSED_FILES, RAW_FILES
from solution.src.utils import save_refined, print_fancy_progress

# 📦 Import atomic cleaning functions and utils
from nb_01_cleaning import (
    REMAINING_FILES,
    optimize_dtypes,
    get_memory_usage_mb,
    clean_string_columns,
    standardize_dataset,
    resolve_content_duplicates,
    print_cleaning_report
)

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print(f"Saving to:    {PROCESSED_DATA_DIR}")

# Global storage for ID remapping (to be applied to relational tables)
GLOBAL_ID_MAP = {}


Saving to:    C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed


## 1. Characters (`characters.csv`)
- Drop rows with missing `character_mal_id`.
- Fill missing `favorites` with 0.
- Drop `about` column (mostly empty/text heavy text not needed for general stats).

In [3]:
# 1. Load
df_chars = load_raw('characters')

# 2. Clean (Conservative)
initial_rows = len(df_chars)

# Remove exact duplicates only
df_chars, std_metrics = standardize_dataset(df_chars)

# Resolve Content Duplicates (Merge IDs)
df_chars, dup_metrics = resolve_content_duplicates(df_chars, primary_key='character_id')

# Store ID map for later use in relational tables
if dup_metrics['id_map']:
    GLOBAL_ID_MAP['character_id'] = dup_metrics['id_map']

# Drop empty rows but NOT just missing IDs
df_chars.dropna(how='all', inplace=True)

df_chars = df_chars.drop(columns=['about'])

rows_dropped_total = initial_rows - len(df_chars)

# 3. Report
report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_chars),
    "Total Dropped": rows_dropped_total
}

print_cleaning_report("Characters", report)

# 4. Optimize & Save
df_chars = optimize_dtypes(df_chars)
save_refined(df_chars, PROCESSED_DATA_DIR / "characters.parquet", format='parquet')

# 5. Capture Valid Character IDs
valid_character_ids = set(df_chars['character_id'].dropna())
print(f"Captured {len(valid_character_ids)} valid Character IDs.")


Loading characters.csv with PyArrow...
--- Characters Report ---
  Initial Rows: 209963
  Standardization: {'duplicates_dropped': 456}
  Content Resolution: {'duplicates_found': 0, 'duplicates_dropped': 0, 'id_map': {}}
  Final Rows: 209507
  Total Dropped: 456
Memory optimization: 74.48 MB -> 73.28 MB (1.6% reduction)
   Changes: character_id: float64->Int32, favorites: float64->Int32
Saving to C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed\characters.parquet (parquet)...
Save complete.
Captured 209506 valid Character IDs.


## 2. Anime Details (`details.csv`)
- **Year**: Derive from `start_date` where missing.
- **Dates**: Convert to datetime objects.
- **Missing Stats**: Keep `score`, `rank` as NaN (preserve "unranked" status).
- **Columns**: Keep `demographics` and others even if they look empty (might have sparse data).

In [4]:
# 1. Load
df_details = load_raw('details')

# 2. Clean (Conservative)
initial_rows = len(df_details)

# Standardize
df_details, std_metrics = standardize_dataset(df_details)

# Resolve Content Duplicates
# Check if resolve_content_duplicates is imported (it should be from previous patch)
# If not, the notebook run might fail, but we assume previous patch worked.
df_details, dup_metrics = resolve_content_duplicates(df_details, primary_key='anime_id')
if dup_metrics['id_map']:
    GLOBAL_ID_MAP['anime_id'] = dup_metrics['id_map']

df_details.dropna(how='all', inplace=True)

# Features
for col in ['start_date', 'end_date']:
    df_details[col] = pd.to_datetime(df_details[col], errors='coerce')

# --- Imputation: meaningful calculations ---
# Calculate Year from start_date if missing
missing_year_before = df_details['year'].isna().sum() if 'year' in df_details.columns else 0
df_details['year'] = df_details['year'].fillna(df_details['start_date'].dt.year)
missing_year_after = df_details['year'].isna().sum() if 'year' in df_details.columns else 0
year_imputed_count = missing_year_before - missing_year_after
# -------------------------------------------

cat_cols = ['type', 'status', 'source', 'rating', 'season']
for col in cat_cols:
    if col in df_details.columns:
        df_details[col] = df_details[col].astype('category')

report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_details),
    "Year Imputed": year_imputed_count
}

print_cleaning_report("Details", report)

# 3. Optimize & Save
df_details = optimize_dtypes(df_details)
save_refined(df_details, PROCESSED_DATA_DIR / "details.parquet", format='parquet')

# 4. Capture Valid Anime IDs
valid_anime_ids = set(df_details['anime_id'].dropna())
print(f"Captured {len(valid_anime_ids)} valid Anime IDs.")


Loading details.csv with PyArrow...
--- Details Report ---
  Initial Rows: 28955
  Standardization: {'duplicates_dropped': 0}
  Content Resolution: {'duplicates_found': 0, 'duplicates_dropped': 0, 'id_map': {}}
  Final Rows: 28955
  Year Imputed: 21838
Memory optimization: 42.17 MB -> 29.94 MB (29.0% reduction)
   Changes: anime_id: int64->int32, score: float64->float32, scored_by: float64->Int32, rank: float64->Int32, popularity: int64->int16, members: int64->int32, favorites: int64->int32, genres: object->category (card: 931/28955), studios: object->category (card: 1910/28955), themes: object->category (card: 1044/28955), demographics: object->category (card: 9/28955), episodes: float64->Int32, year: float64->Int32, producers: object->category (card: 5306/28955), explicit_genres: object->category (card: 1/28955), licensors: object->category (card: 304/28955), streaming: object->category (card: 434/28955)
Saving to C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed\deta

## 3. Anime Stats (`stats.csv`)
- **Votes**: Fill NaN score votes with 0.
- **Percentages**: Fill NaN percentages with 0.
- **Conserve Rows**: Do not drop rows just because they have no votes.

In [5]:
# 1. Load
df_stats = load_raw('stats')

# 2. Clean
initial_rows = len(df_stats)

df_stats, std_metrics = standardize_dataset(df_stats)
df_stats, dup_metrics = resolve_content_duplicates(df_stats, primary_key='anime_id')
# Merge maps if any found here (unlikely if unique per anime)
if dup_metrics['id_map']:
    if 'anime_id' not in GLOBAL_ID_MAP:
        GLOBAL_ID_MAP['anime_id'] = {}
    GLOBAL_ID_MAP['anime_id'].update(dup_metrics['id_map'])

df_stats.dropna(how='all', inplace=True)

report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_stats)
}
print_cleaning_report("Stats", report)

# 3. Optimize & Save
df_stats = optimize_dtypes(df_stats)
save_refined(df_stats, PROCESSED_DATA_DIR / "stats.parquet", format='parquet')


Loading stats.csv with PyArrow...
--- Stats Report ---
  Initial Rows: 28955
  Standardization: {'duplicates_dropped': 0}
  Content Resolution: {'duplicates_found': 30, 'duplicates_dropped': 16, 'id_map': {np.float64(61643.0): np.float64(61635.0), np.float64(62211.0): np.float64(61653.0), np.float64(62539.0): np.float64(61659.0), np.float64(62274.0): np.float64(61626.0), np.float64(62443.0): np.float64(61619.0), np.float64(61646.0): np.float64(61563.0), np.float64(60569.0): np.float64(58230.0), np.float64(60579.0): np.float64(58230.0), np.float64(45746.0): np.float64(45734.0), np.float64(47151.0): np.float64(47150.0), np.float64(47384.0): np.float64(47383.0), np.float64(47138.0): np.float64(46881.0), np.float64(62558.0): np.float64(62557.0), np.float64(62559.0): np.float64(62557.0), np.float64(60405.0): np.float64(60392.0), np.float64(44117.0): np.float64(44115.0)}}
  Final Rows: 28939
Memory optimization: 6.18 MB -> 3.48 MB (43.8% reduction)
   Changes: anime_id: int64->int32, watchin

## 4. User Ratings (`ratings.csv`)
- **Huge File**: Process carefully.
- **Username**: Drop NaN (orphan votes).
- **Rewatching**: Assume `False` (0) if missing.

In [6]:
chunk_size = 5_000_000
csv_path = get_raw_path('ratings')

print(f"Processing {csv_path} in chunks with Conservative Logic...")

id_map_anime = GLOBAL_ID_MAP.get('anime_id', {})

# Mutable storage for stats since creation and consumption are decoupled
stats = {
    'rows_read': 0,
    'dropped_orphans': 0,
    'fk_remapped': 0
    ,'fk_samples': [] # Store samples here
}

def ratings_generator():
    """Generator that yields cleaned chunks of ratings data."""

    def _count_lines(fname):
        """Fast line counting (approx 12MB buffer)."""

        with open(fname, 'rb') as f:
            lines = 0
            buf_size = 1024 * 1024 * 12 # 12MB buffer for speed
            read_f = f.read
            buf = read_f(buf_size)
            while buf:
                lines += buf.count(b'\n')
                buf = read_f(buf_size)
        return lines

    print("Counting rows for progress estimation...")
    total_rows = _count_lines(csv_path) - 1 # Subtract header
    estimated_total_chunks = (total_rows // chunk_size) + (1 if total_rows % chunk_size > 0 else 0)
    print(f"Estimated total rows: {total_rows:,}, Total chunks: {estimated_total_chunks}")

    with pd.read_csv(csv_path, chunksize=chunk_size) as reader:
        for i, chunk in enumerate(reader, 1):
            print_fancy_progress(i, estimated_total_chunks, f'Chunk {i}: ')

            chunk_len = len(chunk)
            stats['rows_read'] += chunk_len

            # 1. Remap FKs
            if id_map_anime:
                mask = chunk['anime_id'].isin(id_map_anime.keys())
                if mask.any():
                    # Log sample remappings
                    if len(stats['fk_samples']) < 5:
                        mask_sample = mask & ~chunk['anime_id'].isna()

                    # Capture sample for logging (Old -> New)
                    if len(stats['fk_samples']) < 5:
                        # Get old values
                        old_values = chunk.loc[mask, 'anime_id'].head(5)
                        # Get their new mapped values
                        new_values = old_values.map(id_map_anime)
                        # Store pair
                        stats['fk_samples'].append(dict(zip(old_values, new_values)))

                    chunk.loc[mask, 'anime_id'] = chunk.loc[mask, 'anime_id'].map(id_map_anime)
                    stats['fk_remapped'] += mask.sum()

            # 2. Filter Orphans
            chunk = chunk[chunk['anime_id'].isin(valid_anime_ids)]
            stats['dropped_orphans'] += (chunk_len - len(chunk))

            # 3. No Imputation / 4. pre-optimization type mapping (optional, but handled by optimize_dtypes)

            yield chunk

# Chain: Generator -> Optimizer -> Saver
# optimize_dtypes and save_refined now support Iterables
cleaned_chunks = ratings_generator()
optimized_chunks = optimize_dtypes(cleaned_chunks, verbose=False) # verbose=True prints details for all chunks

save_refined(optimized_chunks, PROCESSED_DATA_DIR / "ratings.parquet", format='parquet')

final_report = {
    "Total Rows Read": stats['rows_read'],
    "Orphans Dropped": stats['dropped_orphans'],
    "FKs Remapped": stats['fk_remapped'],
    # We don't have final rows count easily without summing again or adding to stats, 
    # but dropped + final = initial.
    "Final Rows Est.": stats['rows_read'] - stats['dropped_orphans']
}

print_cleaning_report("Ratings", final_report)

print("\n--- FK Remapping Samples (First 5 chunks) ---")
for i, sample in enumerate(stats['fk_samples']):
    print(f"Sample {i+1}: {sample}")

_ = gc.collect()


Processing C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\raw\ratings.csv in chunks with Conservative Logic...
Saving to C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed\ratings.parquet (parquet)...
Counting rows for progress estimation...
Estimated total rows: 124,298,357, Total chunks: 25
Chunk 25:  ------------------------------ 100.0%
Save complete.
--- Ratings Report ---
  Total Rows Read: 124298357
  Orphans Dropped: 307757
  FKs Remapped: 739
  Final Rows Est.: 123990600

--- FK Remapping Samples (First 5 chunks) ---
Sample 1: {61646: 61563.0, 62539: 61659.0, 60579: 58230.0, 60569: 58230.0}
Sample 2: {60579: 58230.0, 62211: 61653.0, 60569: 58230.0, 47384: 47383.0}
Sample 3: {61643: 61635.0, 62274: 61626.0, 60579: 58230.0, 62211: 61653.0, 60569: 58230.0}
Sample 4: {61646: 61563.0, 62539: 61659.0, 60579: 58230.0, 61643: 61635.0, 62274: 61626.0}
Sample 5: {61646: 61563.0, 62539: 61659.0, 60569: 58230.0, 47384: 47383.0}


## 5. Other Files
Processing the remaining files with standard optimizations:
- `person_details.csv`
- `profiles.csv`
- `favs.csv`
- `recommendations.csv`
- Relational tables (`*_works.csv`, `*_nicknames.csv`)

In [7]:
results = []
print(f"Processing {len(REMAINING_FILES)} other files...")

valid_ids_map = {}
valid_ids_map['anime_id'] = valid_anime_ids
# We use 'character_id' because files are renamed to standardize IDs
valid_ids_map['character_id'] = valid_character_ids

for i, dataset_name in enumerate(REMAINING_FILES, 1):
    print(f"Processing {dataset_name}...")
    # 1. Read
    df = load_raw(dataset_name)

    # Stats Capture
    initial_rows = len(df)
    initial_mem = get_memory_usage_mb(df)

    # 2. Clean (Generic Logic Inlined)
    actions = []

    df, metrics = standardize_dataset(df)
    if metrics['duplicates_dropped'] > 0:
        actions.append(f"Deduplicated ({metrics['duplicates_dropped']} rows)")

    df.dropna(how='all', inplace=True)

    if dataset_name == "person_details":
        df['birthday'] = pd.to_datetime(df['birthday'], errors='coerce')
        actions.append("Date Parsing (birthday)")

    if dataset_name == "profiles":
        for date_col in ['joined']:
            df['joined'] = pd.to_datetime(df['joined'], errors='coerce')
            actions.append(f"Date Parsing ({'joined'})")

    # 3. Optimize
    df = optimize_dtypes(df, verbose=True)

    # Capture Final Stats
    final_rows = len(df)
    final_mem = get_memory_usage_mb(df)

    # 4. Save
    out_name = PROCESSED_FILES[dataset_name]
    save_refined(df, PROCESSED_DATA_DIR / out_name, format='parquet')

    results.append({
        'file': dataset_name, 
        'rows_src': initial_rows, 
        'rows_dst': final_rows,
        'mem_src': initial_mem,
        'mem_dst': final_mem,
        'actions': actions
    })

# Print Summary
print(f"\n--- Other Files Cleaning Report ---")
for res in results:
    mem_red = (res['mem_src'] - res['mem_dst']) / res['mem_src'] * 100 if res['mem_src'] > 0 else 0
    print(f"[{res['file']}]")
    print(f"  Rows:   {res['rows_src']} -> {res['rows_dst']}")
    print(f"  Memory: {res['mem_src']:.2f} MB -> {res['mem_dst']:.2f} MB ({mem_red:.1f}% reduction)")
    if res['actions']:
        print(f"  Actions: {', '.join(res['actions'])}")
    print("-" * 30)
print(f"Total files processed: {len(results)}")


Processing 9 other files...
Processing person_details...
Loading person_details.csv with PyArrow...
Memory optimization: 41.06 MB -> 25.29 MB (38.4% reduction)
   Changes: person_id: int64->int32, website_url: object->category (card: 17333/76696), image_url: object->category (card: 28429/76696), given_name: object->category (card: 22094/76696), family_name: object->category (card: 27806/76696), favorites: int64->int32, relevant_location: object->category (card: 20/76696)
Saving to C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed\person_details.parquet (parquet)...
Save complete.
Processing profiles...
Loading profiles.csv with PyArrow...
Memory optimization: 154.62 MB -> 27.43 MB (82.3% reduction)
   Changes: gender: object->category (card: 4/337155), birthday: object->category (card: 11345/337155), location: object->category (card: 22/337155), watching: object->category (card: 943/337155), completed: object->category (card: 3381/337155), on_hold: object->category (car